## Installing Libraries

In [ ]:
!pip install -q datasets

In [ ]:
!pip install -q transformers

## Import the Libraries

In [ ]:
from transformers import (
  DistilBertForQuestionAnswering,
  DistilBertTokenizerFast,
  TrainingArguments,
  Trainer,
  default_data_collator,
  pipeline
)
from datasets import load_dataset
import os
os.environ['WANDB_DISABLED'] = 'true'

## Load Dataset

In [ ]:
qa_dataset = load_dataset('squad')
print(qa_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [ ]:
qa_dataset['train'][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

## Loading the Tokenize

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Data Preprocessing

In [ ]:
from random import sample
def prepare_train_features(examples):
  tokenized_examples = tokenizer(
      examples['question'],
      examples['context'],
      truncation='only_second',
      max_length=384,
      stride=128,
      return_overflowing_tokens=True,
      return_offsets_mapping=True,
      padding='max_length'
  )

  sample_mapping = tokenized_examples.pop('overflow_to_sample_mapping')
  offset_mapping = tokenized_examples.pop('offset_mapping')

  tokenized_examples['start_positions'] = []
  tokenized_examples['end_positions'] = []

  for i, offsets in enumerate(offset_mapping):
    input_ids = tokenized_examples['input_ids'][i]
    cls_index = input_ids.index(tokenizer.cls_token_id)

    sequence_ids = tokenized_examples.sequence_ids(i)
    sample_index = sample_mapping[i]
    answers = examples['answers'][sample_index]

    if len(answers['answer_start']) == 0:
      tokenized_examples['start_positions'].append(cls_index)
      tokenized_examples['end_positions'].append(cls_index)
    else:
      start_char = answers['answer_start'][0]
      end_char = start_char + len(answers['text'][0])

      token_start_index = 0
      while sequence_ids[token_start_index] != 1:
        token_start_index += 1

      token_end_index = len(input_ids) - 1
      while sequence_ids[token_end_index] != 1:
        token_end_index -= 1

      if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
        tokenized_examples['start_positions'].append(cls_index)
        tokenized_examples['end_positions'].append(cls_index)
      else:
        while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
          token_start_index += 1
        tokenized_examples['start_positions'].append(token_start_index - 1)

        while offsets[token_end_index][1] >= end_char:
          token_end_index -= 1
        tokenized_examples['end_positions'].append(token_end_index + 1)

  return tokenized_examples

In [ ]:
tokenized_squad = qa_dataset.map(
    prepare_train_features,
    batched=True,
    remove_columns=qa_dataset['train'].column_names
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

## Loading the Pre-trained Model

In [ ]:
model = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Fine-tunning the Pre-trained Model

In [ ]:
from torch.serialization import save
#Define the training arguements
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=1,              # total number of training epochs
    per_device_train_batch_size=32,  # batch size per device during training
    per_device_eval_batch_size=32,   # batch size for evaluation
    fp16=True,
    logging_steps=100,
    log_level = 'warning',
    save_strategy = 'epoch',
    eval_strategy = 'epoch',
    load_best_model_at_end=True,
)

In [ ]:
#Define the trainer
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=tokenized_squad['train'],         # training dataset
    eval_dataset=tokenized_squad['validation'],            # evaluation dataset
    data_collator=default_data_collator,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.167003,1.129100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2767, training_loss=1.4631482349391616, metrics={'train_runtime': 923.603, 'train_samples_per_second': 95.846, 'train_steps_per_second': 2.996, 'total_flos': 8674451270424576.0, 'train_loss': 1.4631482349391616, 'epoch': 1.0})

## Saving the Fine-tuned Model

In [ ]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Testing the Fine_tuned Model

In [ ]:
question = '	What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?'

In [ ]:
context = """
LCMV is most commonly recognized as causing neurological disease,
as its name implies, though infection without symptoms or mild febrile illnesses are
more common clinical manifestations. For infected persons who do become ill,
onset of symptoms usually occurs 8-13 days after exposure to the virus as part of
a biphasic febrile illness. This initial phase, which may last as long as a week,
typically begins with any or all of the following symptoms: fever, malaise,
lack of appetite, muscle aches, headache, nausea, and vomiting. Other symptoms
appearing less frequently include sore throat, cough, joint pain, chest pain,
testicular pain, and parotid (salivary gland) pain. Following a few days of recovery,
a second phase of illness may occur. Symptoms may consist of meningitis
(fever, headache, stiff neck, etc.), encephalitis (drowsiness, confusion,
sensory disturbances, and/or motor abnormalities, such as paralysis),
or meningoencephalitis (inflammation of both the brain and meninges).
LCMV has also been known to cause acute hydrocephalus (increased fluid on the brain),
which often requires surgical shunting to relieve increased intracranial pressure.
In rare instances, infection results in myelitis (inflammation of the spinal cord)
and presents with symptoms such as muscle weakness, paralysis, or changes in body
sensation. An association between LCMV infection and myocarditis (inflammation of
the heart muscles) has been suggested. Previous observations show that most patients
who develop aseptic meningitis or encephalitis due to LCMV survive. No chronic
infection has been described in humans, and after the acute phase of illness,
the virus is cleared from the body. However, as in all infections of the central
nervous system, particularly encephalitis, temporary or permanent neurological damage
is possible. Nerve deafness and arthritis have been reported. Women who become
infected with LCMV during pregnancy may pass the infection on to the fetus.
Infections occurring during the first trimester may result in fetal death and
pregnancy termination, while in the second and third trimesters, birth defects
can develop. Infants infected In utero can have many serious and permanent birth
defects, including vision problems, mental retardation, and hydrocephaly
(water on the brain). Pregnant women may recall a flu-like illness during pregnancy,
or may not recall any illness. LCM is usually not fatal. In general, mortality is
less than 1%.
"""

In [ ]:
pipe = pipeline('question-answering', './results', tokenizer=tokenizer)
pipe(question=question, context=context)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'score': 0.7255641624797136,
 'start': 1285,
 'end': 1341,
 'answer': 'muscle weakness, paralysis, or changes in body\nsensation'}